# 🌱 Bean Leaf Disease Classification — Machine Learning Pipeline

**Problem.** Beans are a key source of protein in East Africa, and two leaf diseases, **angular leaf spot** and **bean rust**, can destroy a large part of a harvest. A farmer should be able to take a photo of a leaf with a phone and get an instant diagnosis.

**Dataset.** *iBean* by the Makerere University AI Lab and the National Crops Resources Research Institute (NaCRRI), Uganda. Field photos of bean leaves taken with smartphones and labelled by agricultural experts, in **3 classes**: `angular_leaf_spot`, `bean_rust`, `healthy` (1,295 images, 500×500 px).

**Notebook outline**
1. Download the data
2. Organise the data into project folders (train / retrain pool / validation / test)
3. Explore the data (data insights)
4. Preprocessing and data augmentation
5. Baseline: vanilla CNN (no optimisation)
6. Optimised model: **pre-trained MobileNetV2** (transfer learning) + data augmentation + dropout + **L2 regularization** + **Adam optimizer** + **early stopping** + learning-rate scheduling + **hyper-parameter tuning** + fine-tuning
7. Evaluation: **Accuracy, Loss, Precision, Recall, F1 score, ROC-AUC**, confusion matrix, classification report
8. Save the model (it becomes the **pre-trained model** that the app's retraining pipeline fine-tunes)
9. Demo samples and retraining data
10. Download the outputs

> **How to run:** open in Google Colab → *Runtime → Change runtime type → T4 GPU* → *Runtime → Run all* (about 15–25 min).
> At the end, download `bean_project_outputs.zip` and this notebook with its outputs (*File → Download → .ipynb*).

In [ ]:
import os, io, json, time, shutil, zipfile, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, log_loss, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')

IMG_SIZE = 224
CLASS_NAMES = ['angular_leaf_spot', 'bean_rust', 'healthy']
DISPLAY = {'angular_leaf_spot': 'Angular leaf spot', 'bean_rust': 'Bean rust', 'healthy': 'Healthy'}
PROJECT = 'bean_project'          # everything the app needs is written here, then zipped
os.makedirs(PROJECT, exist_ok=True)

print('TensorFlow', tf.__version__, '| Keras', getattr(keras, '__version__', '?'))
print('GPU:', tf.config.list_physical_devices('GPU') or 'none found (training will be slower)')

## 1. Download the data
The official files are downloaded from the iBean storage bucket; if that fails the same dataset is fetched from Hugging Face (`AI-Lab-Makerere/beans`).

In [ ]:
RAW = 'raw'
SPLITS = ['train', 'validation', 'test']

def download_from_google_storage():
    for split in SPLITS:
        path = f'{RAW}/{split}.zip'
        urllib.request.urlretrieve(f'https://storage.googleapis.com/ibeans/{split}.zip', path)
        with zipfile.ZipFile(path) as z:
            z.extractall(f'{RAW}/{split}_extracted')

def download_from_hugging_face():
    import subprocess, sys
    try:
        import datasets
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    from datasets import load_dataset
    ds = load_dataset('AI-Lab-Makerere/beans')
    for split in SPLITS:
        col = 'labels' if 'labels' in ds[split].features else 'label'
        names = ds[split].features[col].names
        for i, ex in enumerate(ds[split]):
            cls = names[ex[col]]
            folder = f'{RAW}/{split}_extracted/{split}/{cls}'
            os.makedirs(folder, exist_ok=True)
            ex['image'].convert('RGB').save(f'{folder}/{cls}_{split}.{i}.jpg', quality=95)

def collect(split):
    # list of (file path, label index) found under raw/<split>_extracted/**/<class name>/
    items = []
    for dirpath, _, files in os.walk(f'{RAW}/{split}_extracted'):
        cls = os.path.basename(dirpath)
        if cls in CLASS_NAMES:
            items += [(os.path.join(dirpath, f), CLASS_NAMES.index(cls))
                      for f in sorted(files) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    return items

os.makedirs(RAW, exist_ok=True)
if not all(collect(s) for s in SPLITS):
    try:
        download_from_google_storage(); source = 'Google Cloud Storage (ibeans)'
    except Exception as e:
        print('Google Storage download failed:', e, '-> using Hugging Face')
        shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
        download_from_hugging_face(); source = 'Hugging Face (AI-Lab-Makerere/beans)'
else:
    source = 'already downloaded'

raw = {s: collect(s) for s in SPLITS}
print('Source:', source)
for s in SPLITS:
    print(f'{s:<11} {len(raw[s]):>5} images')
sizes = pd.Series([Image.open(p).size for p, _ in raw['train'][:200]]).value_counts()
print('Original image sizes (first 200 train images):', dict(sizes))

## 2. Organise the data into project folders
* The official **train** split is divided (stratified) into **train (75 %)**, used to build the model here, and a **retrain pool (25 %)** that plays the role of *new data* uploaded later through the app to **retrain** the model.
* The official **validation** split is used for early stopping and hyper-parameter tuning; the **test** split only for the final evaluation.
* Images are resized from 500×500 to the model input size, **224×224**, and saved as JPEG. This keeps the repository small (~20 MB) and the app uses exactly the same files.

In [ ]:
DATA = f'{PROJECT}/data'

def load_image(source, size=IMG_SIZE):
    # identical to src/preprocessing.py :: load_image  (EXIF rotation -> RGB -> resize -> uint8 array)
    img = Image.open(io.BytesIO(source)) if isinstance(source, (bytes, bytearray)) else Image.open(source)
    img = ImageOps.exif_transpose(img).convert('RGB').resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def export(items, split):
    for path, label in items:
        folder = f'{DATA}/{split}/{CLASS_NAMES[label]}'
        os.makedirs(folder, exist_ok=True)
        name = os.path.splitext(os.path.basename(path))[0] + '.jpg'
        Image.fromarray(load_image(path)).save(f'{folder}/{name}', quality=92)

shutil.rmtree(DATA, ignore_errors=True)
train_items, pool_items = train_test_split(raw['train'], test_size=0.25, random_state=SEED,
                                           stratify=[label for _, label in raw['train']])
export(train_items, 'train')
export(pool_items, 'retrain_pool')
export(raw['validation'], 'validation')
export(raw['test'], 'test')

def load_image_folder(folder):
    # identical to src/preprocessing.py :: load_image_folder
    X, y, paths = [], [], []
    for cls in sorted(os.listdir(folder)):
        if cls not in CLASS_NAMES:
            continue
        for f in sorted(os.listdir(f'{folder}/{cls}')):
            X.append(load_image(f'{folder}/{cls}/{f}')); y.append(CLASS_NAMES.index(cls)); paths.append(f'{folder}/{cls}/{f}')
    return np.stack(X), np.array(y), paths

X_train, y_train, train_paths = load_image_folder(f'{DATA}/train')
X_pool,  y_pool,  _           = load_image_folder(f'{DATA}/retrain_pool')
X_val,   y_val,   _           = load_image_folder(f'{DATA}/validation')
X_test,  y_test,  test_paths  = load_image_folder(f'{DATA}/test')

summary = pd.DataFrame({split: np.bincount(y, minlength=3) for split, y in
                        [('train', y_train), ('retrain_pool', y_pool), ('validation', y_val), ('test', y_test)]},
                       index=CLASS_NAMES)
summary.loc['TOTAL'] = summary.sum()
summary

## 3. Explore the data (data insights)

In [ ]:
ax = summary.drop('TOTAL').plot(kind='bar', figsize=(9, 4), rot=0, colormap='viridis')
ax.set_title('Insight 1 — Number of images per class and split'); ax.set_ylabel('Images')
plt.show()
counts = summary.loc[CLASS_NAMES, 'train']
print(f'The classes are balanced (between {counts.min()} and {counts.max()} training images per class), '
      'so accuracy is a fair metric and no class weighting / re-sampling is needed.')

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
for r, cls in enumerate(CLASS_NAMES):
    idx = np.where(y_train == r)[0][:6]
    for c, i in enumerate(idx):
        axes[r, c].imshow(X_train[i]); axes[r, c].axis('off')
    axes[r, 0].set_title(DISPLAY[cls], loc='left', fontsize=12, fontweight='bold')
plt.suptitle('Insight 2 — Example leaves of each class'); plt.tight_layout(); plt.show()
print('Angular leaf spot shows brown angular patches limited by the veins; bean rust shows many small')
print('reddish-brown dots (pustules); healthy leaves are uniformly green. Backgrounds (soil, hands, other')
print('leaves) vary a lot, which is why data augmentation and a pre-trained model help.')

In [ ]:
def colour_stats(X):
    f = X.reshape(len(X), -1, 3).astype(np.float32)
    rgb = f.mean(axis=1)
    return pd.DataFrame({'red': rgb[:, 0], 'green': rgb[:, 1], 'blue': rgb[:, 2],
                         'greenness': rgb[:, 1] / rgb.sum(axis=1),          # share of green in the image
                         'brightness': rgb.mean(axis=1)})

stats = colour_stats(X_train); stats['class'] = [DISPLAY[CLASS_NAMES[i]] for i in y_train]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=stats, x='class', y='greenness', hue='class', legend=False, ax=ax[0], palette='Greens')
ax[0].set_title('Insight 3 — Greenness (share of green in the image)')
sns.kdeplot(data=stats, x='brightness', hue='class', ax=ax[1], fill=True, common_norm=False)
ax[1].set_title('Insight 4 — Image brightness distribution')
plt.tight_layout(); plt.show()
print(stats.groupby('class')[['red', 'green', 'blue', 'greenness', 'brightness']].mean().round(3))

## 4. Preprocessing and data augmentation
| Step | Where | Why |
|---|---|---|
| EXIF rotation, RGB conversion | `load_image()` | phone photos may be rotated / have an alpha channel |
| Resize 500×500 → 224×224 | `load_image()` | input size of MobileNetV2 |
| Train / validation / test split | section 2 | honest evaluation, early stopping |
| **Data augmentation**: random flips, rotation, zoom, contrast | layers inside the model (training only) | leaves can appear at any angle; reduces over-fitting on a small dataset |
| Rescale pixels 0–255 → −1…1 | `Rescaling` layer inside the model | the scaling MobileNetV2 was pre-trained with; being inside the model guarantees the app does the same |

In [ ]:
X_train_f, X_val_f, X_test_f = (a.astype('float32') for a in (X_train, X_val, X_test))
print('Training array :', X_train_f.shape, X_train_f.dtype, f'pixel range {X_train_f.min():.0f}-{X_train_f.max():.0f}')
print('Validation     :', X_val_f.shape, '  Test:', X_test_f.shape)

def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ], name='augmentation')

aug = make_augmentation()
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(X_train[0]); axes[0].set_title('original'); axes[0].axis('off')
for k in range(1, 6):
    out = aug(X_train_f[:1], training=True)[0].numpy()
    axes[k].imshow(np.clip(out, 0, 255).astype('uint8')); axes[k].set_title(f'augmented {k}'); axes[k].axis('off')
plt.suptitle('Data augmentation: the same leaf seen 5 different ways'); plt.show()

def compute_metrics(model, X, y):
    proba = model.predict(X, verbose=0)
    pred = proba.argmax(axis=1)
    return {'Accuracy': accuracy_score(y, pred),
            'Loss': log_loss(y, np.clip(proba, 1e-7, 1), labels=[0, 1, 2]),
            'Precision': precision_score(y, pred, average='macro', zero_division=0),
            'Recall': recall_score(y, pred, average='macro', zero_division=0),
            'F1 score': f1_score(y, pred, average='macro', zero_division=0),
            'ROC AUC': roc_auc_score(y, proba, multi_class='ovr')}

## 5. Baseline: vanilla CNN (no optimisation)
A small convolutional network trained from scratch with plain SGD, **no** pre-training, **no** augmentation, **no** regularization, **no** early stopping and **no** tuning. This is the reference that the optimised model must beat.

In [ ]:
vanilla = keras.Sequential([
    keras.Input((IMG_SIZE, IMG_SIZE, 3)),
    layers.Rescaling(1. / 255),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='vanilla_cnn')
vanilla.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),
                loss='sparse_categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
h_vanilla = vanilla.fit(X_train_f, y_train, validation_data=(X_val_f, y_val), epochs=15, batch_size=32, verbose=2)
print(f'Trained in {time.time() - t0:.0f}s')
vanilla_val = compute_metrics(vanilla, X_val_f, y_val)
vanilla_test = compute_metrics(vanilla, X_test_f, y_test)
pd.Series(vanilla_val, name='vanilla (validation)').round(4)

## 6. Optimised model: transfer learning with MobileNetV2

| Optimisation technique | Implementation |
|---|---|
| **Pre-trained model** | MobileNetV2 trained on ImageNet (1.4 M images) used as feature extractor, then partly fine-tuned |
| **Data augmentation** | random flip / rotation / zoom / contrast layers |
| **Regularization** | Dropout before the classifier + **L2 weight penalty** on the classifier |
| **Optimizer** | **Adam** (adaptive learning rate) |
| **Early stopping** | stop when validation loss has not improved for 5 epochs and restore the best weights |
| **Learning-rate scheduling** | `ReduceLROnPlateau` halves the learning rate when validation loss stalls |
| **Hyper-parameter tuning** | grid search over learning rate × dropout rate, selected on the validation set |
| **Fine-tuning** | second phase: unfreeze the top 40 layers of MobileNetV2 and train with a very small learning rate |

In [ ]:
def build_model(learning_rate=1e-3, dropout=0.3, l2=1e-4):
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False                                   # phase 1: keep the pre-trained features frozen
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')
    x = make_augmentation()(inputs)                          # active only during training
    x = layers.Rescaling(1. / 127.5, offset=-1, name='scale_to_minus1_1')(x)
    x = base(x, training=False)                              # BatchNorm layers stay in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax',
                           kernel_regularizer=regularizers.l2(l2), name='classifier')(x)
    model = keras.Model(inputs, outputs, name='bean_disease_mobilenetv2')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, base

# ---- Hyper-parameter tuning (grid search, scored on the validation set) ----
grid = [{'learning_rate': lr, 'dropout': d} for lr in [1e-3, 3e-4] for d in [0.2, 0.5]]
results = []
t0 = time.time()
for params in grid:
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m, _ = build_model(**params)
    h = m.fit(X_train_f, y_train, validation_data=(X_val_f, y_val), epochs=10, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])
    results.append({**params, 'epochs_run': len(h.history['loss']),
                    'best_val_accuracy': max(h.history['val_accuracy']), 'best_val_loss': min(h.history['val_loss'])})
    print(params, '-> val_accuracy %.4f  val_loss %.4f' % (results[-1]['best_val_accuracy'], results[-1]['best_val_loss']))
print(f'Grid search finished in {time.time() - t0:.0f}s')
tuning = pd.DataFrame(results).sort_values(['best_val_accuracy', 'best_val_loss'], ascending=[False, True])
best_params = {'learning_rate': float(tuning.iloc[0]['learning_rate']), 'dropout': float(tuning.iloc[0]['dropout'])}
print('Best hyper-parameters:', best_params)
tuning

In [ ]:
# ---- Phase 1: train the classifier on top of the frozen pre-trained model ----
keras.backend.clear_session(); tf.random.set_seed(SEED)
model, base = build_model(**best_params)
model.summary()
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]
h1 = model.fit(X_train_f, y_train, validation_data=(X_val_f, y_val), epochs=30, batch_size=32,
               callbacks=callbacks, verbose=2)

In [ ]:
# ---- Phase 2: fine-tune the top 40 layers of MobileNetV2 with a very small learning rate ----
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Trainable weights now:', sum(int(np.prod(w.shape)) for w in model.trainable_weights))
h2 = model.fit(X_train_f, y_train, validation_data=(X_val_f, y_val), epochs=15, batch_size=32,
               callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)],
               verbose=2)

In [ ]:
hist = {k: h1.history[k] + h2.history[k] for k in ['loss', 'val_loss', 'accuracy', 'val_accuracy']}
switch = len(h1.history['loss'])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, metric in zip(ax, ['loss', 'accuracy']):
    a.plot(hist[metric], label=f'train {metric}'); a.plot(hist[f'val_{metric}'], label=f'validation {metric}')
    a.axvline(switch - 0.5, ls='--', color='grey', label='start fine-tuning'); a.set_xlabel('Epoch'); a.legend()
ax[0].set_title('Loss per epoch'); ax[1].set_title('Accuracy per epoch')
plt.tight_layout(); plt.show()
print(f'Phase 1 ran {switch} epochs, phase 2 ran {len(h2.history["loss"])} epochs (early stopping decides when to stop).')

## 7. Evaluation
All metrics on the **validation** set (used for model selection) and on the untouched **test** set.

In [ ]:
opt_val = compute_metrics(model, X_val_f, y_val)
opt_test = compute_metrics(model, X_test_f, y_test)
comparison = pd.DataFrame({'Vanilla CNN (validation)': vanilla_val, 'Optimised (validation)': opt_val,
                           'Vanilla CNN (test)': vanilla_test, 'Optimised (test)': opt_test}).round(4)
comparison

In [ ]:
y_proba = model.predict(X_test_f, verbose=0)
y_pred = y_proba.argmax(axis=1)
print('FINAL TEST-SET RESULTS (optimised model)')
for k, v in opt_test.items():
    print(f'  {k:<10} {v:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0],
            xticklabels=[DISPLAY[c] for c in CLASS_NAMES], yticklabels=[DISPLAY[c] for c in CLASS_NAMES])
ax[0].set_title('Confusion matrix (test set)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
per_class = pd.DataFrame({'Precision': precision_score(y_test, y_pred, average=None),
                          'Recall': recall_score(y_test, y_pred, average=None),
                          'F1 score': f1_score(y_test, y_pred, average=None)}, index=[DISPLAY[c] for c in CLASS_NAMES])
per_class.plot(kind='bar', rot=0, ax=ax[1], ylim=(0.5, 1.02), colormap='viridis'); ax[1].set_title('Per-class metrics (test set)')
plt.tight_layout(); plt.show()

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f'{len(wrong)} of {len(y_test)} test images misclassified')
if len(wrong):
    n = min(len(wrong), 6)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3), squeeze=False)
    for a, i in zip(axes[0], wrong[:n]):
        a.imshow(X_test[i]); a.axis('off')
        a.set_title(f'true: {DISPLAY[CLASS_NAMES[y_test[i]]]}\npred: {DISPLAY[CLASS_NAMES[y_pred[i]]]} ({y_proba[i].max():.0%})', fontsize=9, color='red')
    plt.suptitle('Misclassified test images'); plt.tight_layout(); plt.show()

## 8. Save the model
`models/bean_model.keras` is loaded by the web app for **prediction** and is the **pre-trained model** that the retraining pipeline (`src/retrain.py`) fine-tunes when new data is uploaded.

In [ ]:
MODELS = f'{PROJECT}/models'
os.makedirs(MODELS, exist_ok=True)
model.save(f'{MODELS}/bean_model.keras')

labels = [0, 1, 2]
metadata = {
    'version': 'v1',
    'architecture': 'MobileNetV2 (ImageNet pre-trained) + augmentation + dropout + L2 dense classifier',
    'class_names': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'best_params': best_params,
    'tensorflow_version': tf.__version__,
    'n_training_images': int(len(y_train)),
    'test_metrics': {
        'accuracy': round(float(opt_test['Accuracy']), 4), 'loss': round(float(opt_test['Loss']), 4),
        'precision': round(float(opt_test['Precision']), 4), 'recall': round(float(opt_test['Recall']), 4),
        'f1_score': round(float(opt_test['F1 score']), 4), 'roc_auc': round(float(opt_test['ROC AUC']), 4),
        'per_class_f1': [round(float(v), 4) for v in f1_score(y_test, y_pred, average=None, labels=labels)],
        'confusion_matrix': confusion_matrix(y_test, y_pred, labels=labels).tolist(),
    },
    'vanilla_test_metrics': {k: round(float(v), 4) for k, v in vanilla_test.items()},
}
with open(f'{MODELS}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
with open(f'{MODELS}/training_history.json', 'w') as f:
    json.dump({'phase1_epochs': switch, **{k: [float(v) for v in vals] for k, vals in hist.items()}}, f)

# Check that the saved file loads and gives identical predictions
reloaded = keras.models.load_model(f'{MODELS}/bean_model.keras')
same = np.allclose(reloaded.predict(X_test_f[:16], verbose=0), model.predict(X_test_f[:16], verbose=0), atol=1e-5)
print('Saved model reloads correctly:', same)
print('Model file size: %.1f MB' % (os.path.getsize(f'{MODELS}/bean_model.keras') / 1e6))
print(json.dumps({k: v for k, v in metadata['test_metrics'].items() if k != 'confusion_matrix'}, indent=2))

## 9. Demo samples and retraining data
* `demo_samples/` gets 3 **test** images per class that the model classifies correctly with high confidence; the true class is in the file name. Use them in the video for the prediction demo.
* `demo_samples/new_data_for_retraining.zip` contains the **retrain pool** (images the model has never seen), in one folder per class. Upload it on the app's *Retrain* page.

In [ ]:
DEMO = f'{PROJECT}/demo_samples'
shutil.rmtree(DEMO, ignore_errors=True); os.makedirs(DEMO)
conf = y_proba.max(axis=1)
picked = []
for label, cls in enumerate(CLASS_NAMES):
    ok = [i for i in np.argsort(-conf) if y_test[i] == label and y_pred[i] == label][:3]
    for k, i in enumerate(ok):
        shutil.copy(test_paths[i], f'{DEMO}/true_{cls}_{k + 1}.jpg'); picked.append(i)

with zipfile.ZipFile(f'{DEMO}/new_data_for_retraining.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for cls in CLASS_NAMES:
        for f in sorted(os.listdir(f'{DATA}/retrain_pool/{cls}')):
            z.write(f'{DATA}/retrain_pool/{cls}/{f}', arcname=f'{cls}/{f}')
print(sorted(os.listdir(DEMO)))

# Prediction function used by the app, tested on the demo images
def predict_file(path):
    proba = reloaded.predict(load_image(path)[None].astype('float32'), verbose=0)[0]
    return CLASS_NAMES[int(proba.argmax())], float(proba.max())

files = sorted(f for f in os.listdir(DEMO) if f.endswith('.jpg'))
fig, axes = plt.subplots(1, len(files), figsize=(2.2 * len(files), 2.8))
for a, f in zip(axes, files):
    cls, p = predict_file(f'{DEMO}/{f}')
    true = f[len('true_'):].rsplit('_', 1)[0]
    a.imshow(Image.open(f'{DEMO}/{f}')); a.axis('off')
    a.set_title(f'true {DISPLAY[true]}\npred {DISPLAY[cls]} {p:.0%}', fontsize=8, color='green' if cls == true else 'red')
plt.tight_layout(); plt.show()

## 10. Download the outputs
`bean_project_outputs.zip` contains `data/`, `models/` and `demo_samples/`. Unzip it **into the root of your project repository**. Then also download this notebook with its outputs (*File → Download → Download .ipynb*) and save it as `notebook/bean_disease_classifier.ipynb`.

In [ ]:
shutil.make_archive('bean_project_outputs', 'zip', PROJECT)
print('Created bean_project_outputs.zip (%.1f MB)' % (os.path.getsize('bean_project_outputs.zip') / 1e6))
try:
    from google.colab import files
    files.download('bean_project_outputs.zip')
except ImportError:
    print('Not running in Colab: the zip is in the current folder.')

## Conclusion
* The optimised model (pre-trained MobileNetV2 + augmentation + dropout + L2 + Adam + early stopping + LR scheduling + grid search + fine-tuning) is compared with a vanilla CNN on **6 metrics**: accuracy, loss, precision, recall, F1 score and ROC-AUC (see the comparison table in section 7).
* The saved model is served by the web app for prediction and is the pre-trained starting point that the retraining pipeline fine-tunes on newly uploaded images.